# Model comparison

Runs every available chat-capable model on the same fixed set of 10 representative SMARTS patterns, using identical PubChem/few-shot context for all of them -- so the only thing that varies between columns is the underlying model. There's no automated scoring here; the output is a side-by-side table meant to be read by eye to pick a default model for the pipeline.

In [ ]:
import asyncio

import pandas as pd

from smarts_llm_annotator import llm_utils, name_formatting, prompting, pubchem_lookup

## Benchmark set

10 representative SMARTS: common, easily-recognizable functional groups plus a few genuinely complex real structural-alert patterns (pulled from `data/smarts_examples.csv`) so the harder end reflects real difficulty, not just toy cases.

In [ ]:
BENCHMARK_SMARTS = pd.DataFrame({
    "smarts": [
        "c1ccccc1",                              # benzene ring
        "C(=O)N",                                # amide bond
        "[N+](=O)[O-]",                          # nitro group
        "C(=O)Cl",                               # acid chloride
        "S(=O)(=O)N",                            # sulfonamide
        "C#N",                                   # nitrile
        "C(=O)O",                                # carboxylic acid
        "[$([CH]),$(CC)]#CS(=O)(=O)[C,c]",       # michael acceptor (from smarts_examples.csv)
        "[NX3&!R][NX2&!R]=[NX2&!R]",             # triazenes (from smarts_examples.csv)
        "N#C[#7!$(N(C#N)=C(N)NC)]",              # cyanamide (from smarts_examples.csv)
    ]
})
BENCHMARK_SMARTS

## Discover chat-capable models

The API's model list includes non-chat models (embeddings, rerankers, audio transcription) alongside chat models. There's no reliable standard-API way to filter by capability, so this is a name-pattern heuristic -- edit `_SKIP_PATTERNS` (or just edit `CANDIDATE_MODELS` directly afterward) if it misses something or over-filters.

In [ ]:
ALL_MODELS = asyncio.run(llm_utils.list_available_models())

_SKIP_PATTERNS = ("embed", "rerank", "whisper", "all-proxy-models")
CANDIDATE_MODELS = [m for m in ALL_MODELS if not any(p in m.lower() for p in _SKIP_PATTERNS)]

print(f"{len(CANDIDATE_MODELS)} / {len(ALL_MODELS)} models will be benchmarked:")
print(CANDIDATE_MODELS)

## Build shared context once

PubChem lookup + few-shot example selection + prompt construction don't depend on which model will answer -- computing this once and reusing it for every model keeps the comparison fair (identical prompts) and avoids redundant, rate-limited PubChem calls.

In [ ]:
context_df = pubchem_lookup.lookup_smallest_mw_from_smarts(
    BENCHMARK_SMARTS,
    smarts_col="smarts",
    limit_per_smarts=1000,
    max_workers=8,
    show_progress=True,
    ignore_time_window=True,  # set False for a production run
)
context_df["similar_examples"] = prompting.get_top_n_most_similar_smarts_description_examples_wrapper(
    context_df, "smarts", n=10
)
context_df["prompt"] = context_df.apply(
    lambda row: prompting.build_prompt(row["best_IUPAC_names"], row["similar_examples"], row["smarts"]),
    axis=1,
)
context_df[["smarts", "best_IUPAC_names"]]

## Run every candidate model on the same prompts

Models are run one at a time (not all concurrently) -- each model's 10 SMARTS are still processed concurrently internally, but we don't want to burst ~10x the number of models worth of simultaneous requests at the API at once.

In [ ]:
async def run_all_models(context_df, models):
    columns = {}
    for model in models:
        print(f"Running {model}...")
        out = await llm_utils.run_llm_on_dataframe(context_df, model_name=model, prompt_column="prompt")
        columns[model] = out["llm_output"].apply(name_formatting.normalize_name).values
    return columns

model_columns = asyncio.run(run_all_models(context_df, CANDIDATE_MODELS))

comparison = pd.DataFrame({"smarts": context_df["smarts"].values, **model_columns})
comparison

## Save for manual review

No automated scoring -- open this in a spreadsheet (or just read the table above) and eyeball which column consistently produces the best names. Add your own rating columns/notes directly in the CSV if useful.

In [ ]:
comparison.to_csv("benchmarks/model_comparison_results.csv", index=False)